# TP7 — Préparer du texte : NLTK, spaCy et l'effet sur un modèle
**ÉNONCÉ** · Durée estimée : **une demi-journée**

## 🎯 Ce que vous allez apprendre
- Pourquoi un modèle ne sait pas lire du texte brut, et ce que veut dire **pré-traiter**
- Découper en mots, retirer les **mots vides**, ramener les mots à une forme de base : **racine (NLTK)** contre **lemme (spaCy)**
- **Extraire des entités** (personnes, lieux, organisations) et repérer les **ambiguïtés**
- Transformer le texte en nombres (**TF-IDF**) et **mesurer honnêtement** ce que le prétraitement apporte

## 🧰 Boîte à outils
Noms de fonctions et de classes utiles. **Aucun exemple d'appel n'est donné** : cherchez dans la documentation (scikit-learn, Keras, NumPy, matplotlib) comment les utiliser.

`RegexpTokenizer`  ·  `tokenize`  ·  `PorterStemmer`  ·  `stem`  ·  `nlp.Defaults.stop_words`  ·  `token.lemma_`  ·  `token.is_alpha`  ·  `token.is_stop`  ·  `doc.ents`  ·  `ent.label_`  ·  `spacy.explain`  ·  `nlp.pipe`  ·  `TfidfVectorizer`  ·  `LogisticRegression`  ·  `Counter`

## 📘 Comment travailler
- Les cellules **déjà écrites** sont à lire et à exécuter.
- Les cellules **🎯 Étape** sont à **écrire vous-même**, à partir de la consigne.
- Après chaque étape, une cellule **✅ Contrôle** vérifie votre résultat. Respectez les **noms de variables** demandés.
- Les questions **✍️** demandent une réponse écrite : c'est là que vous montrez que vous avez compris.
- Bloqué ? Relisez l'indice, puis cherchez dans la documentation. Un message d'erreur se lit : la dernière ligne dit souvent ce qui ne va pas.

In [41]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import spacy
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
nlp = spacy.load("en_core_web_sm", disable=["parser"])

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)

## Étape 1 — Les données : 2 400 résumés de films, horreur ou comédie ?
Un **résumé de film** est du **texte brut** : ce qu'on appelle une donnée **non structurée** (pas de colonnes, pas de nombres). Notre but : deviner si le film est une **horreur (1)** ou une **comédie (0)** à partir de son seul résumé. Le fichier contient 1 200 textes de chaque sorte, donc une classification **équilibrée**.

Si vous n'avez pas encore installé les outils : `pip install spacy nltk` puis `python -m spacy download en_core_web_sm`.

In [2]:
import os

DOSSIER_SOURCE = "/mnt/c/Users/Utilisateur/TP-CODE/DeepLearning_RNN_Moons/data_src/"

chemin_csv_films = os.path.join(DOSSIER_SOURCE, "textes_films_horreur_comedie.csv")

In [3]:
df = pd.read_csv(chemin_csv_films)
print(df.shape, "| horreur (1) :", int(df["cible"].sum()), "| comédie (0) :", int((df["cible"] == 0).sum()))
for i in (0, 1, 2):
    print("•", "HORREUR" if df["cible"][i] else "COMÉDIE", "|", df["resume"][i][:220], "...\n")

(2400, 3) | horreur (1) : 1200 | comédie (0) : 1200
• COMÉDIE | Roberto has a new job as the teacher at a nursery school. The first child he meets is Gianluigi, who's mute. Roberto is spirited, bringing a TV into class, then a donkey. He takes children on an unscheduled field trip to ...

• HORREUR | Five years after killing the goalie hockey-masked killer Jason Voorhees, Tommy Jarvis has grown up in various mental hospitals unable to get over the nightmares about Jason's return. When Tommy is sent to a rural halfway ...

• COMÉDIE | Everybody knows that your life is a story. But what if a story was your life? Harold Crick is your average IRS agent: monotonous, boring, and repetitive. But one day this all changes when Harold begins to hear an author  ...



## 🎯 Étape 2 — Étape 1 du prétraitement : découper en mots (tokenisation)
Un modèle ne lit pas des phrases : il compte des **mots**. La première opération consiste à **découper** le texte en unités, appelées **tokens**. On passe aussi en **minuscules**, pour que « Killer » et « killer » soient le même mot : c'est de la **normalisation**.

**À faire :** Prenez le **premier résumé** de `df` (`texte`). Avec un **`RegexpTokenizer` de NLTK** qui ne garde que les suites de lettres (`[a-zA-Z]+`), découpez-le en mots, **en minuscules**. Stockez la liste dans `mots` et affichez le nombre de mots et les 15 premiers.

> 💡 **Indice** : `RegexpTokenizer(r"[a-zA-Z]+")` puis la méthode `.tokenize(...)`. Pour les minuscules : `.lower()` sur le texte.

In [4]:
texte = df["resume"].iloc[0]
# TODO : créer decoupeur et mots
decoupeur = RegexpTokenizer(r"[a-zA-Z]+")
mots = decoupeur.tokenize(texte.lower())
mots[:15]

['roberto',
 'has',
 'a',
 'new',
 'job',
 'as',
 'the',
 'teacher',
 'at',
 'a',
 'nursery',
 'school',
 'the',
 'first',
 'child']

In [5]:
# ✅ Contrôle (ne pas modifier)
controle(len(mots) > 20 and all(m == m.lower() for m in mots), 'Texte découpé en %d mots, tous en minuscules' % len(mots), 'Attendu : une liste de mots en minuscules')

✅ Texte découpé en 122 mots, tous en minuscules


## 🎯 Étape 3 — Étape 2 : retirer les mots vides (stop words)
Dans un texte, beaucoup de mots (« the », « of », « and », « he »…) sont partout, dans les horreurs comme dans les comédies : ils n'aident pas à les distinguer. On les appelle des **mots vides**. Les retirer **réduit le bruit**.

**À faire :** `spaCy` fournit une liste de mots vides anglais : `nlp.Defaults.stop_words`. Stockez-la dans `mots_vides`, puis gardez dans `mots_utiles` uniquement les mots de `mots` qui n'y figurent pas. Affichez le nombre de mots avant et après.

> 💡 **Indice** : Une liste en compréhension : `[m for m in mots if m not in mots_vides]`.

In [6]:
# TODO : mots_vides et mots_utiles
mots_vides = nlp.Defaults.stop_words
mots_utiles = [m for m in mots if m not in mots_vides]
mots_utiles[:15]

['roberto',
 'new',
 'job',
 'teacher',
 'nursery',
 'school',
 'child',
 'meets',
 'gianluigi',
 's',
 'mute',
 'roberto',
 'spirited',
 'bringing',
 'tv']

In [7]:
# ✅ Contrôle (ne pas modifier)
controle(len(mots_utiles) < len(mots) and 'the' not in mots_utiles, 'Mots vides retirés : %d -> %d mots' % (len(mots), len(mots_utiles)), 'Attendu : moins de mots qu\'avant, et plus de « the »')

✅ Mots vides retirés : 122 -> 63 mots


## 🎯 Étape 4 — Étape 3 : ramener les mots à une racine avec NLTK (stemming)
« kill », « killed », « killer » parlent de la même chose. Le **stemming** coupe mécaniquement les fins de mots pour obtenir une **racine** (qui n'est pas toujours un vrai mot). C'est rapide, mais « bête » : il ne connaît pas la grammaire.

**À faire :** Avec le **`PorterStemmer` de NLTK**, calculez la racine de chaque mot de la phrase ci-dessous. Stockez la liste dans `racines` et affichez les couples (mot, racine).

> 💡 **Indice** : `PorterStemmer()` puis `.stem(mot)`.

In [8]:
phrase = "The killers ran away while he was studying better methods"
mots_phrase = phrase.lower().split()
# TODO : racineur et racines
racineur = PorterStemmer()
racines = [racineur.stem(mot) for mot in mots_phrase]

In [9]:
for mot, racine in zip(mots_phrase, racines):
    print(f"{mot} -> {racine}")

the -> the
killers -> killer
ran -> ran
away -> away
while -> while
he -> he
was -> wa
studying -> studi
better -> better
methods -> method


In [10]:
# ✅ Contrôle (ne pas modifier)
controle(racines[mots_phrase.index('studying')] == 'studi' and racines[mots_phrase.index('ran')] == 'ran', 'Racines calculées (remarquez « studi » et « ran »)', 'Attendu : studying -> studi, et ran reste ran')

✅ Racines calculées (remarquez « studi » et « ran »)


### ✍️ Question
**Regardez « studying → studi » et « ran → ran ». Qu'est-ce que le stemming sait faire, et qu'est-ce qu'il ne sait pas faire ?**

*Votre réponse (2 à 4 lignes) :*

> Le stemming sait couper mécaniquement les suffixes réguliers : « studying » → « studi », « killers » → « killer » : les variantes d'un même mot sont regroupées sous une racine commune, rapidement et sans dictionnaire. Mais il ne comprend ni la grammaire ni les formes irrégulières : « ran » reste « ran » car ce n'est pas « run » + un suffixe, le stemmer ne fait pas le lien avec « run ». La racine obtenue n'est d'ailleurs pas toujours un vrai mot (« studi »).

> Pourquoi : le PorterStemmer applique des règles de découpe de fins de mots (étapes 1a/1b : -s, -ing, -ed, y → i…). Ça marche pour les flexions régulières, mais « ran » est un passé irrégulier; aucun suffixe à couper, donc aucun lien avec « run ».

## 🎯 Étape 5 — Étape 4 : ramener les mots à leur forme de base avec spaCy (lemmatisation)
La **lemmatisation** fait la même chose en **comprenant la grammaire** : le **lemme** est le mot du dictionnaire (« ran » → « run », « was » → « be »). C'est plus lent mais plus juste.

**À faire :** Passez la même `phrase` dans spaCy (`doc = nlp(phrase)`). Construisez un tableau pandas **`tableau`** de 10 lignes avec 3 colonnes : `mot`, `racine NLTK` (réutilisez `racines`) et `lemme spaCy`. Affichez-le.

> 💡 **Indice** : Pour chaque `token` de `doc` : `token.text` et `token.lemma_`.

In [ ]:
# TODO : doc et tableau
doc = nlp(phrase)
tableau = pd.DataFrame({
    'mot': [token.text for token in doc],
    'racine NLTK': racines,
    'lemme spaCy': [token.lemma_ for token in doc],
})
tableau

,mot,racine NLTK,lemme spaCy
0,The,the,the
1,killers,killer,killer
2,ran,ran,run
3,away,away,away
4,while,while,while
5,he,he,he
6,was,wa,be
7,studying,studi,study
8,better,better,well
9,methods,method,method


In [12]:
# ✅ Contrôle (ne pas modifier)
controle(tableau.shape == (10, 3) and tableau.loc[tableau['mot'] == 'ran', 'lemme spaCy'].iloc[0] == 'run' and tableau.loc[tableau['mot'] == 'was', 'lemme spaCy'].iloc[0] == 'be', 'Tableau correct : ran -> run, was -> be', 'Attendu 10 lignes, avec ran -> run et was -> be')

✅ Tableau correct : ran -> run, was -> be


### ✍️ Question
**Comparez les colonnes « racine NLTK » et « lemme spaCy ». Dans quel cas préféreriez-vous l'une ou l'autre ?**

*Votre réponse (2 à 4 lignes) :*

> Je vois que lemme a fait lui meme le lower. Le NLTK renvoi une racine qui n'est pas toujours un mot mais suffisement clair et proche pour etre comprise par une IA. Le lemme est un mot qui existe et au plus simple grammaticalement parlant, ce qui est plus juste dans le cas où le sens du mot est important.

## 🎯 Étape 6 — Étape 5 : extraire les entités nommées
Une **entité nommée** est un nom propre de personne, de lieu, d'organisation, une date, un nombre… spaCy les repère automatiquement. Extraire les entités permet de savoir **de qui, d'où, de quand** parle un texte, et de créer des variables utiles.

**À faire :** 1) Passez le **premier résumé** dans spaCy et affichez ses entités (texte → type, avec la signification du type). 2) Passez les **300 premiers résumés** dans `nlp.pipe(...)` et comptez les **types d'entités** dans `types` (un `Counter`).

> 💡 **Indice** : `for ent in doc.ents:` donne `ent.text` et `ent.label_`. `spacy.explain("GPE")` donne la signification. `nlp.pipe(liste_de_textes, batch_size=64)` traite beaucoup de textes plus vite qu'un par un.

In [22]:
# TODO : afficher les entités du 1er résumé, puis calculer docs300 et types
doc = nlp(texte)
for ent in doc.ents:
    print(f"Text : {ent.text} → Type : {ent.label_}; Signification : ({spacy.explain(ent.label_)}")

Text : Roberto → Type : PERSON; Signification : (People, including fictional
Text : first → Type : ORDINAL; Signification : ("first", "second", etc.
Text : Gianluigi → Type : PERSON; Signification : (People, including fictional
Text : Roberto → Type : PERSON; Signification : (People, including fictional
Text : Isabella → Type : PERSON; Signification : (People, including fictional
Text : one → Type : CARDINAL; Signification : (Numerals that do not fall under another type
Text : Isabella → Type : PERSON; Signification : (People, including fictional
Text : Sardinia → Type : GPE; Signification : (Countries, cities, states
Text : Roberto → Type : PERSON; Signification : (People, including fictional
Text : ten → Type : CARDINAL; Signification : (Numerals that do not fall under another type
Text : Gianluigi → Type : PERSON; Signification : (People, including fictional
Text : Sardinia → Type : GPE; Signification : (Countries, cities, states
Text : Gianluigi → Type : PERSON; Signification : (Pe

In [30]:
docs300 = list(nlp.pipe(df['resume'][:300], batch_size=64))
types = Counter()

for doc in docs300:
    for ent in doc.ents:
        types.update([ent.label_])

In [31]:
display(types)

Counter({'PERSON': 1067,
         'ORG': 275,
         'GPE': 227,
         'DATE': 178,
         'CARDINAL': 152,
         'NORP': 76,
         'ORDINAL': 36,
         'TIME': 32,
         'WORK_OF_ART': 31,
         'LOC': 23,
         'PRODUCT': 18,
         'FAC': 8,
         'MONEY': 7,
         'QUANTITY': 3,
         'EVENT': 3,
         'LAW': 1,
         'LANGUAGE': 1})

In [32]:
# ✅ Contrôle (ne pas modifier)
controle('PERSON' in types and sum(types.values()) > 300, 'Entités extraites : %d dans 300 résumés' % sum(types.values()), 'Attendu des entités de type PERSON et plus de 300 au total')

✅ Entités extraites : 2138 dans 300 résumés


## 🎯 Étape 7 — Étape 6 : chasser les ambiguïtés
Une extraction automatique n'est **pas parfaite** : un même nom peut recevoir **plusieurs étiquettes** selon le contexte (un prénom pris pour une organisation, un lieu pour une personne…). Le référentiel parle de **réduction des ambiguïtés** : il faut d'abord **les repérer**.

**À faire :** À partir de `docs300`, construisez le dictionnaire `etiquettes` (clé = texte de l'entité, valeur = **ensemble** des types qu'elle a reçus), puis `ambigus` : celles qui ont **au moins 2 types différents**. Affichez le nombre d'entités ambiguës et **10 exemples**.

> 💡 **Indice** : `etiquettes.setdefault(ent.text, set()).add(ent.label_)`.

In [ ]:
# Dictionnaire : texte de l'entité -> ensemble des types reçus
# (un set, car une même entité ne doit pas compter 2 fois le même type)
etiquettes = {}

# Parcours des 300 documents traités par spaCy
for doc in docs300:
    # doc.ents = entités nommées trouvées dans ce résumé
    for ent in doc.ents:
        # setdefault : récupère l'ensemble existant, ou en crée un vide si la clé est nouvelle
        # .add : ajoute le type de l'entité (doublons automatiquement ignorés)
        etiquettes.setdefault(ent.text, set()).add(ent.label_)

# Garde uniquement les entités vues avec AU MOINS 2 types différents
# -> ce sont les ambiguïtés (ex. "Apple" = ORG ou PERSON selon le contexte)
ambigus = {texte: labels for texte, labels in etiquettes.items() if len(labels) > 1}

# Affichage du bilan
print(f"Nombre d'entités ambiguës : {len(ambigus)}")
print("10 exemples :")

# list(...) pour pouvoir découper, [:10] pour les 10 premiers, enumerate inutile ici mais gardé
for i, (texte, labels) in enumerate(list(ambigus.items())[:10]):
    print(f"  {texte} : {labels}")

Nombre d'entités ambiguës : 34
10 exemples :
  Pierre : {'PERSON', 'ORG'}
  Julia : {'ORG', 'GPE'}
  Channard : {'PERSON', 'ORG'}
  Cronos : {'PERSON', 'ORG'}
  Mercedes : {'PERSON', 'ORG'}
  Alice : {'PERSON', 'ORG'}
  Steven : {'PERSON', 'ORG'}
  Tillie : {'PERSON', 'GPE'}
  Sunny : {'PERSON', 'ORG'}
  Ace Ventura : {'PERSON', 'GPE'}


In [34]:
# ✅ Contrôle (ne pas modifier)
controle(len(ambigus) > 0, '%d entités ambiguës repérées' % len(ambigus), 'Attendu au moins une entité avec plusieurs types')

✅ 34 entités ambiguës repérées


### ✍️ Question
**Quelles ambiguïtés avez-vous trouvées ? Que feriez-vous pour fiabiliser ces extractions avant d'entraîner un modèle ?**

*Votre réponse (2 à 4 lignes) :*

> 34 Ambiguitées trouvées, toutes sur des noms propres qui sont soit prises pour des PERSON soit pour des ORGanisations. Pour fiabiliser les extractions, il faudrait peut-être ajouter des règles de filtrage pour les noms propres, ou bien utiliser un modèle de NLP plus avancé pour mieux distinguer les entités de type PERSON et ORG.

## 🎯 Étape 8 — Étape 7 : le pipeline complet
On assemble tout : un texte brut entre, un **texte propre** sort, en minuscules, **sans ponctuation, sans mots vides, avec des lemmes**.

**À faire :** Écrivez `nettoyer_doc(doc)` qui reçoit un document spaCy et renvoie une chaîne : les **lemmes en minuscules** des tokens qui sont **alphabétiques** (`is_alpha`) et **pas des mots vides** (`is_stop`), séparés par des espaces. Appliquez-la à **tous** les résumés dans une colonne `df["propre"]` (⏱️ environ 20 secondes).

> 💡 **Indice** : `nlp.pipe(df["resume"], batch_size=128)` donne les documents un par un. Les attributs : `token.lemma_`, `token.is_alpha`, `token.is_stop`.

In [36]:
docs = nlp.pipe(df["resume"], batch_size=128)

def nettoyer_doc(doc):
    # liste des lemmes en minuscules, gardés s'ils sont alphabétiques et pas des mots vides
    lemmes = []
    for token in doc:
        if token.is_alpha and not token.is_stop:
            lemmes.append(token.lemma_.lower())
    # on les rassemble en une seule chaîne séparée par des espaces
    return " ".join(lemmes)

df["propre"] = [nettoyer_doc(doc) for doc in docs]

df["propre"][:10]

0    roberto new job teacher nursery school child m...
1    year kill goalie hockey mask killer jason voor...
2    everybody know life story story life harold cr...
3    pierre professional dancer suffer heart diseas...
4    retire detective insomnia call solve series mu...
5    little deaths compose segment short story sex ...
6    death blind sara hang basement house twin sist...
7    day job new protector metro city megamind mini...
8    film center wound gulf war veteran return nati...
9    old man son walk road suddenly meet speak crow...
Name: propre, dtype: str

In [37]:
# ✅ Contrôle (ne pas modifier)
controle('propre' in df.columns and df['propre'].notna().all() and ',' not in df['propre'].iloc[0] and df['propre'].str.len().min() > 0, 'Colonne propre créée pour les %d textes' % len(df), 'Vérifiez nettoyer_doc : pas de ponctuation, aucun texte vide')

✅ Colonne propre créée pour les 2400 textes


## Étape 9 — Deux autres versions du texte, pour comparer
On prépare aussi une version « **minuscules** » et une version « **racines NLTK** » (avec les outils des étapes 1 à 3), pour comparer les traitements à l'étape 9.

In [39]:
def racines_texte(texte):
    return " ".join(racineur.stem(m) for m in decoupeur.tokenize(texte.lower()) if m not in mots_vides)

df["minuscules"] = df["resume"].str.lower()
df["racines"] = df["resume"].apply(racines_texte)
print(df[["resume", "minuscules", "racines", "propre"]].iloc[0].apply(lambda s: s[:150]).to_string())

resume        Roberto has a new job as the teacher at a nurs...
minuscules    roberto has a new job as the teacher at a nurs...
racines       roberto new job teacher nurseri school child m...
propre        roberto new job teacher nursery school child m...


## 🎯 Étape 10 — Étape 8 : transformer le texte en nombres, et entraîner
Pour qu'un modèle lise un texte, on le transforme en **nombres** : le **TF-IDF** donne à chaque mot un poids élevé s'il est **fréquent dans ce texte** et **rare dans les autres** (« killer » aide ; « the » n'aide pas). Le **vocabulaire** (la liste des mots connus) **s'apprend sur l'apprentissage seul**, sinon le test « souffle » des informations : c'est une **fuite de données**.

**À faire :** 1) Découpez `df` en `train` / `test` : **25 % de test**, **stratifié sur `cible`**, `random_state=42`. 2) Pour chaque colonne **`resume`** (texte brut) et **`propre`** (texte nettoyé) : apprenez un `TfidfVectorizer` **sur `train` seulement**, entraînez une `LogisticRegression(max_iter=2000)`, puis calculez sur `test` la **précision**, le **F1** et la **taille du vocabulaire**. Rangez dans un dictionnaire `resultats` (clés `"resume"` et `"propre"`) et affichez-le en tableau.

> 💡 **Indice** : `vec.fit_transform(train[col])` sur l'apprentissage, `vec.transform(test[col])` sur le test. `X.shape[1]` = nombre de mots du vocabulaire.

In [47]:
# 1) Découpage train/test : 25 % de test, stratifié sur 'cible' (même proportion horreur/comédie
#    dans les deux ensembles), random_state=42 pour la reproductibilité
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["cible"])

# 2) Deux vectoriseurs TF-IDF indépendants (un par version du texte) :
#    chacun apprendra son propre vocabulaire
vec_resume = TfidfVectorizer()
vec_propre = TfidfVectorizer()

# --- Version 'resume' (texte brut) ---
# fit_transform sur train UNIQUEMENT : le vocabulaire s'apprend sur l'apprentissage seul
# (fit sur test = fuite de données, le test "soufflerait" des informations)
X_train_resume = vec_resume.fit_transform(train['resume'])
# transform sur test : on applique le vocabulaire appris, sans réapprendre
X_test_resume = vec_resume.transform(test['resume'])

# Modèle de référence : régression logistique (max_iter=2000 pour assurer la convergence)
logr_resume = LogisticRegression(max_iter=2000)
logr_resume.fit(X_train_resume, train['cible'])
pred_resume = logr_resume.predict(X_test_resume)

# Métriques sur le test + taille du vocabulaire (X.shape[1] = nombre de mots connus)
resultats = {
    "resume": {
        "précision": accuracy_score(test['cible'], pred_resume),
        "F1": f1_score(test['cible'], pred_resume, average='macro'),
        "mots du vocabulaire": X_train_resume.shape[1]
    }
}

# --- Version 'propre' (texte nettoyé : lemmes, minuscules, sans mots vides) ---
# Même protocole, avec son propre vectoriseur
X_train_propre = vec_propre.fit_transform(train['propre'])
X_test_propre = vec_propre.transform(test['propre'])

logr_propre = LogisticRegression(max_iter=2000)
logr_propre.fit(X_train_propre, train['cible'])
pred_propre = logr_propre.predict(X_test_propre)

resultats["propre"] = {
    "précision": accuracy_score(test['cible'], pred_propre),
    "F1": f1_score(test['cible'], pred_propre, average='macro'),
    "mots du vocabulaire": X_train_propre.shape[1]
}

In [48]:
# Affichage sous forme de tableau
pd.DataFrame(resultats).T


,précision,F1,mots du vocabulaire
resume,0.856667,0.856641,17025.0
propre,0.863333,0.863309,13550.0


In [49]:
# ✅ Contrôle (ne pas modifier)
controle(len(train) == 1800 and 0.7 < resultats['resume']['précision'] < 0.95 and resultats['propre']['mots du vocabulaire'] < resultats['resume']['mots du vocabulaire'], 'Comparaison faite : vocabulaire réduit à %d mots' % resultats['propre']['mots du vocabulaire'], 'Attendu : 1800 textes d\'apprentissage, précisions entre 0.7 et 0.95, vocabulaire réduit après nettoyage')

✅ Comparaison faite : vocabulaire réduit à 13550 mots


## Étape 11 — Étape 9 : le prétraitement améliore-t-il vraiment ? Mesurons sur 5 découpages
Un seul découpage ne prouve rien : le hasard du découpage fait varier le résultat. On **répète 5 fois** avec des découpages différents et on regarde la **moyenne** et l'**écart-type** (± : la dispersion). On le fait aussi avec **200 textes seulement**, pour voir ce qui change quand les données sont rares.

⏱️ Quelques dizaines de secondes.

In [50]:
def evaluer(colonne, taille=None, **options):
    scores, voc = [], []
    for graine in range(5):
        tr, te = train_test_split(df, test_size=0.25, stratify=df["cible"], random_state=graine)
        if taille:
            tr = tr.sample(taille, random_state=graine)
        vec = TfidfVectorizer(**options); X = vec.fit_transform(tr[colonne])
        m = LogisticRegression(max_iter=2000).fit(X, tr["cible"])
        scores.append(np.mean(m.predict(vec.transform(te[colonne])) == te["cible"])); voc.append(X.shape[1])
    return np.mean(scores), np.std(scores), int(np.mean(voc))

configs = [("texte brut (sans minuscules)", "resume", dict(lowercase=False)), ("minuscules", "minuscules", {}),
           ("minuscules + mots vides retirés", "minuscules", dict(stop_words="english")),
           ("racines NLTK + mots vides retirés", "racines", {}), ("lemmes spaCy + mots vides retirés", "propre", {})]
lignes = []
for nom, col, opt in configs:
    m1, s1, v1 = evaluer(col, **opt); m2, s2, _ = evaluer(col, taille=200, **opt)
    lignes.append({"traitement": nom, "précision (1800 textes)": f"{m1:.3f} ± {s1:.3f}", "précision (200 textes)": f"{m2:.3f} ± {s2:.3f}", "mots du vocabulaire": v1})
bilan = pd.DataFrame(lignes)
bilan

,traitement,précision (1800 textes),précision (200 textes),mots du vocabulaire
0,texte brut (sans minuscules),0.843 ± 0.014,0.670 ± 0.046,18754
1,minuscules,0.838 ± 0.018,0.672 ± 0.048,17058
2,minuscules + mots vides retirés,0.872 ± 0.004,0.682 ± 0.070,16774
3,racines NLTK + mots vides retirés,0.872 ± 0.008,0.712 ± 0.066,11926
4,lemmes spaCy + mots vides retirés,0.867 ± 0.006,0.691 ± 0.065,13564


### ✍️ Question
**Quelle étape de prétraitement améliore le plus la précision ? Les racines et les lemmes apportent-ils autre chose ? Que se passe-t-il avec 200 textes seulement ?**

*Votre réponse (2 à 4 lignes) :*

> D'une maniere générale, le retirage des mots vide aide a la précision, ensuite ce serait le NLTK qui offre le meilleur résultat avec un petit corpus. Le lemme spaCy n'a rien apporté de plus que miniscule ou NLTK, voire meme un peu regressé en précision par rapport aux racines. Avec un petit corpus, la precision baisse d'environ 0.15 points, tandis que le bruit explose (de x3 a x10), ce qui est significatif.

## ✅ Ce qu'il faut retenir
- Un modèle ne lit pas du texte : on **découpe** (tokens), on **normalise** (minuscules), on **retire les mots vides**, on ramène les mots à une **forme de base** (racine NLTK ou lemme spaCy), puis on **vectorise** (TF-IDF).
- **Racine** (rapide, approximative, « studi ») contre **lemme** (grammaticalement juste, « study », plus lent).
- Les **entités nommées** donnent le « qui, où, quand », mais l'extraction est **imparfaite** : il faut **chercher les ambiguïtés**.
- Le **vocabulaire** s'apprend **sur l'apprentissage seul**.
- **Mesurer** : un prétraitement améliore parfois nettement (mots vides), parfois seulement la taille du modèle (racines, lemmes), et avec peu de données le hasard masque tout.

## 🧭 Lien avec le référentiel (compétence C.14.3)
| Ce que demande le référentiel | Où dans ce TP |
|---|---|
| Outils NLP comme **SpaCy et NLTK** | NLTK : étapes 1 et 3 · spaCy : étapes 2, 4, 5, 6, 7 |
| **Extraction précise des entités** | Étapes 5 et 6 |
| **Normalisation du texte** | Étapes 1 à 4 et 7 |
| **Réduction des ambiguïtés** | Étape 6 (repérage), étape 3 à 4 (formes d'un même mot) |
| **Optimiser les performances** des modèles | Étapes 8 et 9 |